# Initial CatBoost Baseline

This notebook evaluates a first CatBoost regression baseline for predicting crop yield, represented by `AP Ratio`, on the preprocessed SPAS-Dataset-BD data. The goal of this notebook is not yet final model optimization; it establishes a reproducible IID baseline and checks whether CatBoost can learn useful signal from the cleaned tabular features.


## Architecture Alignment

This notebook implements the first modeling block of the proposed pipeline: an IID CatBoost regression baseline for `AP Ratio`. It does not yet implement the full model zoo, conformal prediction, or OOD evaluation. Those should be built as separate follow-up notebooks once this baseline is understood.


In [1]:
import pandas as pd
import numpy as np
# Load the dataset
df = pd.read_csv('../dataset/preprocessed_data.csv')

## Dataset Load

The notebook loads `preprocessed_data.csv`, which contains the cleaned SPAS records after removing invalid spreadsheet artifacts, dropping `Production` to avoid target leakage, and adding corrected humidity features. The target column remains `AP Ratio`, while categorical variables such as district, season, crop name, and crop calendar fields are kept as strings for CatBoost's native categorical handling.


In [2]:
df

,Area,AP Ratio,District,Season,Avg Temp,Avg Humidity,Crop Name,Transplant,Growth,Harvest,Max Temp,Min Temp,Humidity Min,Humidity Max,Humidity Range
0,177321,0.851027,Bagerhat,Kharif 2,26.0,72.5,Aman,Jun,Jul to Oct,Nov to Dec,40,12.0,60,85,25
1,25646,1.175778,Bandarban,Kharif 2,26.0,72.5,Aman,Jun,Jul to Oct,Nov to Dec,40,12.0,60,85,25
2,231401,0.770589,Barguna,Kharif 2,26.0,72.5,Aman,Jun,Jul to Oct,Nov to Dec,40,12.0,60,85,25
3,302665,0.757104,Barishal,Kharif 2,26.0,72.5,Aman,Jun,Jul to Oct,Nov to Dec,40,12.0,60,85,25
4,388575,1.100652,Bhola,Kharif 2,26.0,72.5,Aman,Jun,Jul to Oct,Nov to Dec,40,12.0,60,85,25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4185,256,2.195312,Sirajganj,Kharif 1,25.0,70.0,Wood Apple,May,Jun to Aug,Aug to Sep,30,20.0,60,80,20
4186,50,1.380000,Sunamganj,Kharif 1,25.0,70.0,Wood Apple,May,Jun to Aug,Aug to Sep,30,20.0,60,80,20
4187,95,1.242105,Sylhet,Kharif 1,25.0,70.0,Wood Apple,May,Jun to Aug,Aug to Sep,30,20.0,60,80,20
4188,497,1.348089,Tangail,Kharif 1,25.0,70.0,Wood Apple,May,Jun to Aug,Aug to Sep,30,20.0,60,80,20


In [3]:
# CatBoost baseline setup

In [8]:
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

## CatBoost Configuration

CatBoost is used because it can consume categorical columns directly through the `cat_features` argument. This avoids one-hot encoding at this stage and gives a strong tabular baseline. The categorical features used here are `District`, `Season`, `Crop Name`, `Transplant`, `Growth`, and `Harvest`.


In [5]:
CAT_FEATURES = [
    "District",
    "Season",
    "Crop Name",
    "Transplant",
    "Growth",
    "Harvest",
]


In [6]:
RANDOM_STATE = 42
DATA_PATH = "../dataset/preprocessed_data.csv"
TARGET_COL = "AP Ratio"

In [7]:
df = pd.read_csv(DATA_PATH)


## IID Split With All Yield Records

The first experiment uses all rows in the preprocessed dataset, including the small number of zero-yield records. The data is split into 70% training, 15% validation, and 15% test. The validation set is used only for early stopping, while the test set is held out for final metric reporting.


In [ ]:
# Leaving all records in for the first baseline, to match the current preprocessed dataset. For a final zero-yield sensitivity run, set DROP_ZERO_YIELD to True and re-run the notebook.
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=RANDOM_STATE,
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
)


In [14]:
model = CatBoostRegressor(
    loss_function="RMSE",
    eval_metric="RMSE",
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=3.0,
    random_seed=RANDOM_STATE,
    allow_writing_files=False,
    verbose=100,
)

model.fit(
    X_train,
    y_train,
    cat_features=CAT_FEATURES,
    eval_set=(X_val, y_val),
    early_stopping_rounds=100,
    use_best_model=True,
)

y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

metrics_all = {
    "setting": "CatBoost, all rows",
    "rows": len(df),
    "rmse": rmse,
    "mae": mae,
    "r2": r2,
    "best_iteration": model.get_best_iteration(),
}

print(f"Rows used: {len(df)}")
print(f"Train/validation/test sizes: {len(X_train)} / {len(X_val)} / {len(X_test)}")
print(f"Best iteration: {model.get_best_iteration()}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"R2: {r2:.4f}")

feature_importance = (
    pd.DataFrame({
        "feature": X.columns,
        "importance": model.get_feature_importance(),
    })
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

print("\nTop feature importances:")
print(feature_importance.head(15).to_string(index=False))


0:	learn: 20.1841717	test: 12.0861061	best: 12.0861061 (0)	total: 115ms	remaining: 1m 55s
100:	learn: 9.7722403	test: 10.9779159	best: 10.8897962 (84)	total: 14s	remaining: 2m 4s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 10.88979622
bestIteration = 84

Shrink model to first 85 iterations.
Rows used: 4190
Train/validation/test sizes: 2933 / 628 / 629
Best iteration: 84
RMSE: 7.8999
MAE: 2.7264
R2: -0.0778

Top feature importances:
       feature  importance
      Max Temp   27.824429
          Area   27.203154
      Avg Temp   18.422875
      District   13.464954
        Growth    3.432420
     Crop Name    2.914933
        Season    1.783326
    Transplant    1.531097
       Harvest    1.113725
  Avg Humidity    1.075933
  Humidity Max    0.880288
  Humidity Min    0.226443
      Min Temp    0.126423
Humidity Range    0.000000


## Nonzero-Yield Sensitivity Run

The second experiment removes rows where `AP Ratio <= 0`. This checks whether zero-yield records are hurting the baseline model. The same 70/15/15 split and CatBoost configuration are then reused so the comparison is controlled.


In [17]:
# For only nonzero yields:
df_nonzero = df[df[TARGET_COL] > 0].copy()
X_red = df_nonzero.drop(columns=[TARGET_COL])
y_red = df_nonzero[TARGET_COL]

X_train_red, X_temp_red, y_train_red, y_temp_red = train_test_split(
    X_red,
    y_red,
    test_size=0.30,
    random_state=RANDOM_STATE,
)

X_val_red, X_test_red, y_val_red, y_test_red = train_test_split(
    X_temp_red,
    y_temp_red,
    test_size=0.50,
    random_state=RANDOM_STATE,
)

In [19]:
model_nonzero = CatBoostRegressor(
    loss_function="RMSE",
    eval_metric="RMSE",
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=3.0,
    random_seed=RANDOM_STATE,
    allow_writing_files=False,
    verbose=100,
)

model_nonzero.fit(
    X_train_red,
    y_train_red,
    cat_features=CAT_FEATURES,
    eval_set=(X_val_red, y_val_red),
    early_stopping_rounds=100,
    use_best_model=True,
)

y_pred_red = model_nonzero.predict(X_test_red)

rmse = np.sqrt(mean_squared_error(y_test_red, y_pred_red))
mae = mean_absolute_error(y_test_red, y_pred_red)
r2 = r2_score(y_test_red, y_pred_red)

metrics_nonzero = {
    "setting": "CatBoost, AP Ratio > 0",
    "rows": len(df_nonzero),
    "rmse": rmse,
    "mae": mae,
    "r2": r2,
    "best_iteration": model_nonzero.get_best_iteration(),
}

print(f"Rows used: {len(df_nonzero)}")
print(f"Train/validation/test sizes: {len(X_train_red)} / {len(X_val_red)} / {len(X_test_red)}")
print(f"Best iteration: {model_nonzero.get_best_iteration()}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"R2: {r2:.4f}")

feature_importance = (
    pd.DataFrame({
        "feature": X_red.columns,
        "importance": model_nonzero.get_feature_importance(),
    })
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

print("\nTop feature importances:")
print(feature_importance.head(15).to_string(index=False))


0:	learn: 20.2310423	test: 6.6661188	best: 6.6661188 (0)	total: 79.6ms	remaining: 1m 19s
100:	learn: 9.9963709	test: 4.9643913	best: 4.9643913 (100)	total: 11.6s	remaining: 1m 42s
200:	learn: 6.6813475	test: 4.7475660	best: 4.7475660 (200)	total: 22.1s	remaining: 1m 27s
300:	learn: 5.3611361	test: 4.8143114	best: 4.7380271 (216)	total: 31.4s	remaining: 1m 12s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 4.738027104
bestIteration = 216

Shrink model to first 217 iterations.
Rows used: 4177
Train/validation/test sizes: 2923 / 627 / 627
Best iteration: 216
RMSE: 10.8358
MAE: 2.2156
R2: 0.2414

Top feature importances:
       feature  importance
          Area   30.262313
      Max Temp   22.520446
      Avg Temp   17.124375
      District   11.628459
        Growth    5.947431
     Crop Name    4.340023
       Harvest    3.850913
        Season    1.590126
    Transplant    1.002842
      Min Temp    0.736832
  Avg Humidity    0.605868
  Humidity Max    0.308618
Humi

## Baseline Sanity Check

A CatBoost model should be compared against a trivial mean predictor on the exact same test split. This sanity check shows whether CatBoost is adding predictive value beyond simply predicting the average training yield.


In [ ]:
mean_pred_red = np.full(shape=len(y_test_red), fill_value=y_train_red.mean(), dtype=float)

mean_rmse = np.sqrt(mean_squared_error(y_test_red, mean_pred_red))
mean_mae = mean_absolute_error(y_test_red, mean_pred_red)
mean_r2 = r2_score(y_test_red, mean_pred_red)

metrics_mean_nonzero = {
    "setting": "Mean baseline, AP Ratio > 0",
    "rows": len(df_nonzero),
    "rmse": mean_rmse,
    "mae": mean_mae,
    "r2": mean_r2,
    "best_iteration": np.nan,
}

pd.DataFrame([metrics_nonzero, metrics_mean_nonzero])


If CatBoost has a positive \(R^2\) while the mean baseline is near or below zero, then CatBoost is learning some signal. However, if CatBoost has high RMSE despite better MAE, the model is likely being punished by a small number of extreme-yield outliers.


## Target Distribution and Outlier Analysis

The baseline results suggest that RMSE is strongly affected by extreme target values. The following cells inspect the distribution of `AP Ratio` and list the largest yield values. This is important because a few very high-yield records can dominate RMSE even when MAE improves.


In [20]:
# Target distribution inspection. The large max value explains why RMSE is sensitive to outliers.
df[TARGET_COL].describe()

count    4177.000000
mean        4.820748
std        17.986051
min         0.013926
25%         1.296296
50%         2.535714
75%         4.726027
max       951.216667
Name: AP Ratio, dtype: float64

In [21]:
df.sort_values(TARGET_COL, ascending=False).head(20)

,Area,AP Ratio,District,Season,Avg Temp,Avg Humidity,Crop Name,Transplant,Growth,Harvest,Max Temp,Min Temp,Humidity Min,Humidity Max,Humidity Range
1906,60,951.216667,Netrokona,Kharif 1,34.5,82.5,Jack Fruit,Apr,No need to do,Apr to Jun,47,22.0,70,95,25
1882,100,286.810000,Jamalpur,Kharif 1,34.5,82.5,Jack Fruit,Apr,No need to do,Apr to Jun,47,22.0,70,95,25
1891,300,275.853333,Kushtia,Kharif 1,34.5,82.5,Jack Fruit,Apr,No need to do,Apr to Jun,47,22.0,70,95,25
1920,94,201.797872,Sirajganj,Kharif 1,34.5,82.5,Jack Fruit,Apr,No need to do,Apr to Jun,47,22.0,70,95,25
1907,170,189.405882,Nilphamari,Kharif 1,34.5,82.5,Jack Fruit,Apr,No need to do,Apr to Jun,47,22.0,70,95,25
1919,80,152.050000,Sherpur,Kharif 1,34.5,82.5,Jack Fruit,Apr,No need to do,Apr to Jun,47,22.0,70,95,25
1867,150,139.380000,Brahmanbaria,Kharif 1,34.5,82.5,Jack Fruit,Apr,No need to do,Apr to Jun,47,22.0,70,95,25
1889,200,128.590000,Kishoreganj,Kharif 1,34.5,82.5,Jack Fruit,Apr,No need to do,Apr to Jun,47,22.0,70,95,25
1883,100,113.110000,Jashore,Kharif 1,34.5,82.5,Jack Fruit,Apr,No need to do,Apr to Jun,47,22.0,70,95,25
1910,250,104.576000,Panchagar,Kharif 1,34.5,82.5,Jack Fruit,Apr,No need to do,Apr to Jun,47,22.0,70,95,25


## Largest Prediction Errors

This residual inspection identifies the test records where the nonzero-yield CatBoost model makes the largest absolute errors. At the current stage, the model is especially inefficient on extreme high-yield cases, particularly Jack Fruit records with unusually large `AP Ratio` values. This indicates that ordinary RMSE training on the raw target is sensitive to outliers and may not be the best formulation for this dataset.

Current takeaway: CatBoost is not inherently unsuitable, but the raw-target CatBoost baseline is inefficient because it underfits or misestimates rare extreme yield cases. The next modeling step should test a log-transformed target, such as `log1p(AP Ratio)`, and compare MAE/RMSE/R2 after converting predictions back to the original scale.


In [22]:
# Inspect the largest test errors for the nonzero-yield model.
results_nonzero = X_test_red.copy()
results_nonzero["y_true"] = y_test_red.values
results_nonzero["y_pred"] = y_pred_red
results_nonzero["abs_error"] = np.abs(results_nonzero["y_true"] - results_nonzero["y_pred"])

results_nonzero.sort_values("abs_error", ascending=False).head(20)

,Area,District,Season,Avg Temp,Avg Humidity,Crop Name,Transplant,Growth,Harvest,Max Temp,Min Temp,Humidity Min,Humidity Max,Humidity Range,y_true,y_pred,abs_error
1891,300,Kushtia,Kharif 1,34.5,82.5,Jack Fruit,Apr,No need to do,Apr to Jun,47,22.0,70,95,25,275.853333,33.536919,242.316414
1910,250,Panchagar,Kharif 1,34.5,82.5,Jack Fruit,Apr,No need to do,Apr to Jun,47,22.0,70,95,25,104.576000,30.026465,74.549535
1918,167,Shariatpur,Kharif 1,34.5,82.5,Jack Fruit,Apr,No need to do,Apr to Jun,47,22.0,70,95,25,7.598802,59.167756,51.568954
1877,100,Feni,Kharif 1,34.5,82.5,Jack Fruit,Apr,No need to do,Apr to Jun,47,22.0,70,95,25,23.610000,66.063261,42.453261
1880,184,Gopalganj,Kharif 1,34.5,82.5,Jack Fruit,Apr,No need to do,Apr to Jun,47,22.0,70,95,25,12.369565,40.117432,27.747867
2595,201,Rajshahi,Rabi,17.5,77.5,Lemon,Jan,Jan to Jan,Jan,25,10.0,60,95,35,18.830846,3.124908,15.705937
1856,136,Sirajganj,Kharif 1,29.0,67.5,Guava,Apr,No need to do,Throuout The year,35,23.0,50,85,35,23.367647,9.035255,14.332392
2039,4,Rajshahi,Kharif 2,30.0,80.0,Jamrul,Jun,Jul to Aug,Jun to Sep,35,25.0,70,90,20,20.500000,7.014246,13.485754
3871,5821,Gaibandha,Kharif 2,29.0,65.0,Sugarcane,Oct,Nov to Aug,Aug to Sep,38,20.0,45,85,40,24.694898,11.795008,12.899890
3666,20,Habiganj,Kharif 1,30.0,80.0,Ripe Papaya,Apr,No need to do,Apr to May,35,25.0,70,90,20,25.500000,12.732048,12.767952
